In [5]:
# System import
import sys
sys.path.insert(0, "..")

# Local import
from utils.data_processing import build_chroma_document_from_mongo_document
from utils.mongo_handler import get_legislation_by_query

In [6]:
query = {"category": "Luật"}
legislations = get_legislation_by_query(query)
chroma_documents = [build_chroma_document_from_mongo_document(doc) for doc in legislations["data"]]
print(f"Number of documents: {len(chroma_documents)}")
print(f"First document: {chroma_documents[0]}")

Number of documents: 307
First document: {'documents': 'QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc Luật số: 42/2024/QH15 Hà Nội, ngày 29 tháng 6 năm 2024 LUẬT QUẢN LÝ, SỬ DỤNG VŨ KHÍ, VẬT LIỆU NỔ VÀ CÔNG CỤ HỖ TRỢ Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam; Quốc hội ban hành Luật Quản lý, sử dụng vũ khí, vật liệu nổ và công cụ hỗ trợ. Chương INHỮNG QUY ĐỊNH CHUNG Điều 1. Phạm vi điều chỉnhLuật này quy định về quản lý, sử dụng vũ khí, vật liệu nổ, tiền chất thuốc nổ, công cụ hỗ trợ; nguyên tắc, trách nhiệm của cơ quan, tổ chức, cá nhân trong quản lý, sử dụng vũ khí, vật liệu nổ, tiền chất thuốc nổ, công cụ hỗ trợ nhằm bảo vệ an ninh quốc gia, bảo đảm trật tự, an toàn xã hội, bảo vệ quyền con người, quyền công dân và phục vụ phát triển kinh tế - xã hội. Điều 2. Giải thích từ ngữTrong Luật này, các từ ngữ dưới đây được hiểu như sau: 1. Vũ khí là thiết bị, phương tiện hoặc tổ hợp những thiết bị, phương tiện được chế tạo, sản xuất có khả năng gây s

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text splitter configuration
chunk_size = 3000  # chunk size (characters)
chunk_overlap = 300  # chunk overlap (characters)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,  
    chunk_overlap=chunk_overlap,  
    add_start_index=True,  # track index in original document
)   

# Sanitizing metadata function
def sanitize_metadata(metadata):
    """Sanitize metadata by removing keys that are not serializable."""
    for k, v in metadata.items():
        if isinstance(v, (dict, list)):
            metadata[k] = ", ".join(map(str, v))  # Convert to string
    return metadata

from langchain_core.documents import Document

# Convert the documents to Chroma Document format
docs = [Document(page_content=doc["documents"], metadata=sanitize_metadata(doc["metadata"])) for doc in chroma_documents]

In [9]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Ollama model initialization
local_model = "llama3"
llm = ChatOllama(model=local_model, base_url="http://ollama:11434")
print(f"LLM model loaded: {llm.model}")

# Embedding model
model_name = "all-MiniLM-L6-v2"  # Example model name, can be changed to any HuggingFace model
# embedding_model = OllamaEmbeddings(model=model_name)
# print(f"Embedding model loaded: {embedding_model.model}")
embedding_model = HuggingFaceEmbeddings(model_name=model_name)
print(f"Embedding model loaded: {embedding_model.model_name}")

# Vector store initialization
persist_directory = f"../database/{model_name}/{chunk_size}_{chunk_overlap}"
vectordb_host = "vector-db"
vectordb_port = 8000
    
vector_store = Chroma(
    # persist_directory=persist_directory,
    host=vectordb_host,
    port=vectordb_port,
    collection_name="legislation",
    embedding_function=embedding_model,
)

LLM model loaded: llama3
Embedding model loaded: all-MiniLM-L6-v2


In [10]:
import time

for doc in docs:
    chunked_docs = text_splitter.split_documents([doc])
    print(f"Number of chunks {doc.metadata['name']} splited into: {len(chunked_docs)}")
    
    for attempt in range(3):  # Try up to 3 times
        try:
            # Operation that might fail
            vector_store.add_documents(
                documents=chunked_docs,
                collection_name="legislation",
                ids=[(str(doc.metadata["id"]) + "_" + str(doc.metadata["start_index"])) for doc in chunked_docs],
            )
            break  # Exit the loop on success
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
            time.sleep(2)  # Wait before retrying
    else:
        print("Failed to add documents after 3 attempts. Skipping this document.")

Number of chunks LUẬT QUẢN LÝ, SỬ DỤNG VŨ KHÍ, VẬT LIỆU NỔ VÀ CÔNG CỤ HỖ TRỢ splited into: 53
Number of chunks LUẬT SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT ĐẤT ĐAI SỐ 31/2024/QH15, LUẬT NHÀ Ở SỐ 27/2023/QH15, LUẬT KINH DOANH BẤT ĐỘNG SẢN SỐ 29/2023/QH15 VÀ LUẬT CÁC TỔ CHỨC TÍN DỤNG SỐ 32/2024/QH15 splited into: 2
Number of chunks LUẬT BẢO HIỂM XÃ HỘI splited into: 68
Number of chunks LUẬT THỦ ĐÔ splited into: 58
Number of chunks LUẬT SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT CẢNH VỆ splited into: 6
Number of chunks LUẬT SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT ĐẤU GIÁ TÀI SẢN splited into: 23
Number of chunks LUẬT TRẬT TỰ, AN TOÀN GIAO THÔNG ĐƯỜNG BỘ splited into: 61
Number of chunks LUẬT ĐƯỜNG BỘ splited into: 55
Number of chunks LUẬT CÔNG NGHIỆP QUỐC PHÒNG, AN NINH VÀ ĐỘNG VIÊN CÔNG NGHIỆP splited into: 42
Number of chunks LUẬT TỔ CHỨC TÒA ÁN NHÂN DÂN splited into: 49
Number of chunks LUẬT LƯU TRỮ splited into: 27
Number of chunks LUẬT CÁC TỔ CHỨC TÍN DỤNG splited into: 109
Number of chunks L

In [11]:
retrieve_result = vector_store.similarity_search(
    query="Các nguyên tắc cơ bản của hôn nhân và gia đình là gì?",
    k=10
)

for i, doc in enumerate(retrieve_result):
    print(f"Document {i + 1}:")
    print("Name:", doc.metadata.get("name", "N/A"))
    print("Content:", doc.page_content)
    print("Id:", doc.metadata.get("id", "N/A"))
    print("Document number:", doc.metadata.get("numberDoc", "N/A"))
    print("Fields:", doc.metadata.get("fields", "N/A"))
    # print("Metadata:", retrieve_result["metadatas"][0][i])
    # print("Score:", retrieve_result["distances"][0][i])
    print("-"* 50)  # Separator for readability

Document 1:
Name: LUẬT SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU CỦA LUẬT THI HÀNH ÁN DÂN SỰ
Content: giá không thành thì Chấp hành viên quyết định giảm giá để tiếp tục bán đấu giá tài sản. 5. Mỗi lần giảm giá theo quy định tại các khoản 1, 3 và 4 Điều này không quá 10% giá khởi điểm của lần bán đấu giá liền kề trước đó.” 37. Sửa đổi, bổ sung các khoản 3, 4, 5 và 6 Điều 106 như sau:“3. Hồ sơ đăng ký chuyển quyền sở hữu, sử dụng gồm có: a) Văn bản đề nghị của cơ quan thi hành án dân sự; b) Bản sao bản án, quyết định; c) Quyết định thi hành án; d) Quyết định kê biên tài sản, nếu có; đ) Văn bản bán đấu giá thành hoặc quyết định giao tài sản, biên bản giao nhận tài sản để thi hành án; e) Giấy tờ khác có liên quan đến tài sản. 4. Trường hợp tài sản là quyền sử dụng đất, nhà ở và tài sản khác gắn liền với đất mà không có hoặc không thu hồi được Giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà ở và tài sản khác gắn liền với đất thì cơ quan có thẩm quyền có trách nhiệm cấp Giấy chứng nhận theo quy định 